# Perceptron Implementation — Activity 2

This notebook explains and practices the Perceptron step-by-step:

1. Separate features and labels
2. Initialize weights
3. Apply the bias trick
4. Calculate the dot product / weighted sum
5. Apply the step function
6. Calculate delta
7. Update weights when the prediction is wrong
8. Repeat for multiple epochs
9. Test the trained Perceptron


In [1]:
# NumPy is used for arrays and mathematical operations.
import numpy as np


## 1. Create a Simple Dataset

Each row contains `[feature_1, feature_2, label]`.

The last column is the target/label:
- `0` = Class 0
- `1` = Class 1

In [2]:
data = np.array([
    [0, 0, 0],
    [0, 1, 0],
    [1, 0, 0],
    [1, 1, 1],
    [2, 1, 1],
    [1, 2, 1]
])

print("Dataset:")
print(data)


Dataset:
[[0 0 0]
 [0 1 0]
 [1 0 0]
 [1 1 1]
 [2 1 1]
 [1 2 1]]


## 2. Perceptron Function

The Perceptron takes the weighted sum of the inputs, uses a step function for a binary prediction, and updates the weights when the prediction is wrong.

General update rule:

$$w_{new} = w_{old} + \alpha(d-y)x$$

In this implementation, `alpha` is not written explicitly, so it is effectively `1`.

In [3]:
def perceptron(data, num_iter):
    # ---------------------------------------------------------
    # STEP 1: Separate FEATURES and LABELS
    # ---------------------------------------------------------
    # Take all rows and all columns except the last column.
    features = data[:, :-1]

    # Take all rows from the last column.
    labels = data[:, -1]

    # ---------------------------------------------------------
    # STEP 2: Initialize weights
    # ---------------------------------------------------------
    # We have 2 features, so we need 2 feature weights.
    # +1 is for the bias weight.
    # Initial weights are all zero.
    w = np.zeros(shape=(1, features.shape[1] + 1))

    # Store the number of mistakes made in every epoch.
    misclassified_ = []

    # ---------------------------------------------------------
    # STEP 3: Repeat training for num_iter epochs
    # ---------------------------------------------------------
    for epoch in range(num_iter):
        # Reset mistake counter for this epoch.
        misclassified = 0

        # -----------------------------------------------------
        # STEP 4: Process one training sample at a time
        # -----------------------------------------------------
        for x, label in zip(features, labels):

            # -------------------------------------------------
            # STEP 5: Bias trick
            # -------------------------------------------------
            # Before: [x1, x2]
            # After : [1, x1, x2]
            # The constant 1 lets the first weight act as bias.
            x = np.insert(x, 0, 1)

            # -------------------------------------------------
            # STEP 6: Calculate weighted sum / dot product
            # -------------------------------------------------
            # y = w0*1 + w1*x1 + w2*x2
            y = np.dot(w, x.transpose())

            # -------------------------------------------------
            # STEP 7: Step function
            # -------------------------------------------------
            # If y > 0 -> prediction = 1
            # Otherwise -> prediction = 0
            target = 1.0 if (y > 0) else 0.0

            # -------------------------------------------------
            # STEP 8: Calculate delta
            # -------------------------------------------------
            # delta = actual label - predicted output
            # delta = 0  -> correct prediction
            # delta = 1  -> predicted 0 but actual is 1
            # delta = -1 -> predicted 1 but actual is 0
            delta = label - target

            # -------------------------------------------------
            # STEP 9: Update weights only if prediction is wrong
            # -------------------------------------------------
            if delta != 0:
                misclassified += 1

                # General rule: w_new = w_old + alpha*delta*x
                # Here alpha = 1, so:
                # w_new = w_old + delta*x
                w += (delta * x)

        # Save mistakes made during this epoch.
        misclassified_.append(misclassified)

    # Return final weights and mistakes per epoch.
    return w, misclassified_


## 3. Train the Perceptron

`num_iter = 10` means the complete dataset is processed 10 times.
Each complete pass through the dataset is called an **epoch**.

In [4]:
num_iter = 10

w, misclassified_ = perceptron(data, num_iter)


## 4. Display the Results

In [5]:
print("Final Weights:", w)
print("Misclassified/epoch:", misclassified_)


Final Weights: [[-2.  2.  1.]]
Misclassified/epoch: [1, 3, 3, 2, 1, 0, 0, 0, 0, 0]


## 5. Understand the Learned Weights

The weights are arranged as:

`[bias_weight, x1_weight, x2_weight]`

In [6]:
print("Bias weight (w0):", w[0][0])
print("Feature x1 weight (w1):", w[0][1])
print("Feature x2 weight (w2):", w[0][2])


Bias weight (w0): -2.0
Feature x1 weight (w1): 2.0
Feature x2 weight (w2): 1.0


## 6. Manually Test One Input

This section shows the same operations used by the Perceptron when making a prediction on a new input.

In [7]:
# New input containing two features.
x = np.array([2, 2])

# Add the bias input of 1.
x_with_bias = np.insert(x, 0, 1)

# Calculate weighted sum.
weighted_sum = np.dot(w, x_with_bias.transpose())

# Apply the step function.
prediction = 1 if weighted_sum > 0 else 0

print("Input:", x)
print("Input after bias trick:", x_with_bias)
print("Weighted Sum:", weighted_sum)
print("Prediction:", prediction)


Input: [2 2]
Input after bias trick: [1 2 2]
Weighted Sum: [4.]
Prediction: 1


## 7. Detailed Learning Process

Run the next cell when you want to see the values of:

`x → weighted sum → prediction → delta → old weights → new weights`

for every training sample.

In [8]:
def perceptron_with_details(data, num_iter):
    # Separate features and labels.
    features = data[:, :-1]
    labels = data[:, -1]

    # Initialize weights to zero.
    w = np.zeros(features.shape[1] + 1)

    print("Initial weights:", w)
    print("=" * 60)

    # Start epochs.
    for epoch in range(num_iter):
        misclassified = 0

        print(f"\nEPOCH {epoch + 1}")
        print("-" * 60)

        # Process every training sample.
        for x, label in zip(features, labels):
            # Add bias.
            x = np.insert(x, 0, 1)

            # Calculate weighted sum.
            y = np.dot(w, x)

            # Step function.
            target = 1.0 if y > 0 else 0.0

            # Calculate delta.
            delta = label - target

            print("\nInput:", x)
            print("Actual label:", label)
            print("Weighted sum:", y)
            print("Prediction:", target)
            print("Delta:", delta)

            # Update only if the prediction is wrong.
            if delta != 0:
                misclassified += 1
                old_w = w.copy()
                w += delta * x

                print("Prediction was WRONG")
                print("Old weights:", old_w)
                print("New weights:", w)
            else:
                print("Prediction was CORRECT")
                print("Weights unchanged:", w)

        print("\nMistakes in this epoch:", misclassified)

    print("\n" + "=" * 60)
    print("FINAL WEIGHTS:", w)

    return w


In [9]:
# Run this cell to observe the complete learning process.
# Start with 3 or 5 epochs so the output is easier to read.

final_w = perceptron_with_details(data, 3)


Initial weights: [0. 0. 0.]

EPOCH 1
------------------------------------------------------------

Input: [1 0 0]
Actual label: 0
Weighted sum: 0.0
Prediction: 0.0
Delta: 0.0
Prediction was CORRECT
Weights unchanged: [0. 0. 0.]

Input: [1 0 1]
Actual label: 0
Weighted sum: 0.0
Prediction: 0.0
Delta: 0.0
Prediction was CORRECT
Weights unchanged: [0. 0. 0.]

Input: [1 1 0]
Actual label: 0
Weighted sum: 0.0
Prediction: 0.0
Delta: 0.0
Prediction was CORRECT
Weights unchanged: [0. 0. 0.]

Input: [1 1 1]
Actual label: 1
Weighted sum: 0.0
Prediction: 0.0
Delta: 1.0
Prediction was WRONG
Old weights: [0. 0. 0.]
New weights: [1. 1. 1.]

Input: [1 2 1]
Actual label: 1
Weighted sum: 4.0
Prediction: 1.0
Delta: 0.0
Prediction was CORRECT
Weights unchanged: [1. 1. 1.]

Input: [1 1 2]
Actual label: 1
Weighted sum: 4.0
Prediction: 1.0
Delta: 0.0
Prediction was CORRECT
Weights unchanged: [1. 1. 1.]

Mistakes in this epoch: 1

EPOCH 2
------------------------------------------------------------

Input: [

## 8. Key Formulas to Remember

### Weighted Sum
$$z = w_0(1) + w_1x_1 + w_2x_2$$

### Step Function
$$y = \begin{cases}1, & z > 0\\0, & z \leq 0\end{cases}$$

### Delta
$$\delta = d-y$$

### Weight Update
$$w_{new} = w_{old} + \alpha\delta x$$

In your code, `alpha = 1`, so:

$$w_{new} = w_{old} + \delta x$$

## 9. Viva Quick Revision

- `features = data[:, :-1]` → gets input features.
- `labels = data[:, -1]` → gets target labels.
- `np.zeros(...)` → initializes weights to zero.
- `np.insert(x, 0, 1)` → adds the bias input.
- `np.dot(w, x)` → calculates weighted sum.
- Step function → converts the result into 0 or 1.
- `delta = label - target` → calculates the error/delta.
- `if delta != 0` → checks for a wrong prediction.
- `w += delta * x` → updates the weights.
- `num_iter` → number of epochs.
- `misclassified_` → stores mistakes in each epoch.